# 🍎 APPLE CNN — MODEL EVALUATION & METRICS
### Project: Plant Disease Detection (Computer Vision)
**Model:** `6th Trained_Model/apple_cnn_best.keras`  
**Test Data:** 15% Held-out split from `3rd Preprocessing/apple_processed_data.npz` (267 images)

In [2]:
# ================================================================
# 🍎 APPLE CNN — EVALUATION & METRICS REPORT
# ================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import csv
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

BASE = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"
if not os.path.exists(BASE):
    BASE = r"G:\My Drive\Plant Disease Detection (Computer Vision)"

MODEL_PATH = os.path.join(BASE, "6th Trained_Model", "apple_cnn_best.keras")
NPZ_PATH   = os.path.join(BASE, "3rd Preprocessing", "apple_processed_data.npz")
EVAL_DIR   = os.path.join(BASE, "5th Model_Evaluation")

CM_PNG_PATH     = os.path.join(EVAL_DIR, "apple_confusion_matrix.png")
REPORT_TXT_PATH = os.path.join(EVAL_DIR, "apple_classification_report.txt")
SUMMARY_JSON    = os.path.join(EVAL_DIR, "apple_evaluation_summary.json")

print("=" * 75)
print("🍎 1. LOADING DATASET & MODEL")
print("=" * 75)

data = np.load(NPZ_PATH)
X = data['X']
y = data['y']
class_names = data['class_names']

# Recreate exact same stratified split
SEED = 42
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

print(f"Test Set shape: {X_test.shape} (dtype: {X_test.dtype})")
print(f"Loading Model from: {MODEL_PATH}")

model = tf.keras.models.load_model(MODEL_PATH)
model.summary()

print("\n" + "=" * 75)
print("🍎 2. EVALUATING TEST SET")
print("=" * 75)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)
print(f"\n🎯 Test Accuracy : {test_acc*100:.2f}%")
print(f"🎯 Test Loss     : {test_loss:.4f}")

# Predictions
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Classification Report
report = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)
report_txt = classification_report(y_test, y_pred, target_names=class_names)
print("\n" + "=" * 75)
print("📊 CLASSIFICATION REPORT")
print("=" * 75)
print(report_txt)

with open(REPORT_TXT_PATH, "w", encoding="utf-8") as f:
    f.write(f"APPLE CNN MODEL EVALUATION\nTest Accuracy: {test_acc*100:.2f}%\nTest Loss: {test_loss:.4f}\n\n")
    f.write(report_txt)

# Confusion Matrix Plot
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f"Apple CNN Confusion Matrix (Accuracy: {test_acc*100:.2f}%)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(CM_PNG_PATH, dpi=300)
plt.close()

# Save JSON Summary
summary = {
    "project": "Plant Disease Detection",
    "plant": "Apple",
    "model": "Apple CNN — MobileNetV2",
    "test_images": len(X_test),
    "test_accuracy": float(test_acc),
    "test_accuracy_percent": float(test_acc * 100),
    "test_loss": float(test_loss),
    "classification_report": report
}

with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

print(f"\n✅ Evaluation reports saved in: {EVAL_DIR}")


Mounted at /content/drive
🍎 1. LOADING DATASET & MODEL
Test Set shape: (267, 224, 224, 3) (dtype: uint8)
Loading Model from: /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/apple_cnn_best.keras


Model: "Apple_MobileNetV2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ apple_augmentation (Sequential) │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,473,931 (24.70 MB)

 Trainable params: 2,025,795 (7.73 MB)

 Non-trainable params: 396,544 (1.51 MB)

 Optimizer params: 4,051,592 (15.46 MB)


🍎 2. EVALUATING TEST SET
9/9 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.9775 - loss: 0.0968

🎯 Test Accuracy : 97.75%
🎯 Test Loss     : 0.0968
9/9 ━━━━━━━━━━━━━━━━━━━━ 13s 1s/step

📊 CLASSIFICATION REPORT
                  precision    recall  f1-score   support

         Healthy       1.00      0.96      0.98        90
      Apple_Scab       0.96      0.99      0.97        90
Cedar_Apple_Rust       0.98      0.99      0.98        87

        accuracy                           0.98       267
       macro avg       0.98      0.98      0.98       267
    weighted avg       0.98      0.98      0.98       267


✅ Evaluation reports saved in: /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/5th Model_Evaluation
